# 10 — Contextual validation (unblind)

After annotations and axes are frozen, **unblind** sampling cells and inspect whether
the same construct means the same thing in high- vs low-rated books.

In [1]:
import json
import sys
from pathlib import Path

import pandas as pd

cwd = Path.cwd().resolve()
root = cwd
for _ in range(6):
    if (root / "configs").is_dir() and (root / "src").is_dir():
        break
    root = root.parent
sys.path.insert(0, str(root))

from src.stage11_refined_construct_analysis.analysis import notebook_helpers as nh

ctx = nh.setup("10_contextual_validation")
cfg = ctx.cfg

Project root : /home/polina/Documents/Cursor_Projects/romantic_novels_large_corpus
Config       : configs/stage11/refined_constructs.yaml
Run          : v4_l12_granular_final_call49
Outputs      : results/stage11_refined_construct_analysis/v4_l12_granular_final_call49/notebook_analysis/10_contextual_validation


## Unblind cell key

In [2]:
cell_key = nh.load_cell_key(cfg)
print(json.dumps(cell_key, indent=2)[:2000])
ctx.save_markdown(json.dumps(cell_key, indent=2), "cell_key_unblinded")

meanings = cfg.section("evidence", "cell_meanings")
cell_tbl = pd.DataFrame(
    [{"cell": k, "meaning": v} for k, v in meanings.items()]
)
display(cell_tbl)
ctx.save_table(cell_tbl, "cell_meanings")

{
  "labels": [
    "CELL_A",
    "CELL_B",
    "CELL_C",
    "CELL_D"
  ],
  "meanings": {
    "CELL_A": "high_prevalence_high_tier",
    "CELL_B": "high_prevalence_low_tier",
    "CELL_C": "low_prevalence_high_tier",
    "CELL_D": "low_prevalence_low_tier"
  },
  "sealed": true,
  "note": "Do not open until notebook 10 contextual validation. Pass A/B must only see CELL_* labels; position/tertile may be visible."
}
  saved markdown: results/stage11_refined_construct_analysis/v4_l12_granular_final_call49/notebook_analysis/10_contextual_validation/tables/cell_key_unblinded.md


,cell,meaning
0,CELL_A,high_prevalence_high_tier
1,CELL_B,high_prevalence_low_tier
2,CELL_C,low_prevalence_high_tier
3,CELL_D,low_prevalence_low_tier


  saved table: results/stage11_refined_construct_analysis/v4_l12_granular_final_call49/notebook_analysis/10_contextual_validation/tables/cell_meanings.csv  (4 rows)


## Construct × rating: code distributions from Pass B (now interpretable)

In [3]:
# For each hypothesis, summarise Pass B dominant codes — cells were blinded during coding.
rows = []
for hyp in ("H1", "H2", "H3", "H4", "H5", "H6"):
    b = nh.load_audit_jsonl(cfg, hyp, "B")
    if b.empty:
        continue
    for _, r in b.iterrows():
        resp = r.get("response") or {}
        if not isinstance(resp, dict):
            resp = {}
        rows.append(
            {
                "hypothesis": hyp,
                "topic_id": r.get("topic_id"),
                "dominant_code": resp.get("dominant_code") or r.get("code"),
                "mixed": resp.get("mixed_topic") or (str(r.get("code")) == "MIXED"),
                "meaning_differs_across_cells": resp.get("meaning_differs_across_cells"),
            }
        )
summary = pd.DataFrame(rows)
if not summary.empty:
    display(
        summary.groupby(["hypothesis", "meaning_differs_across_cells"], dropna=False)
        .size()
        .rename("n")
        .reset_index()
    )
    ctx.save_table(summary, "pass_b_cell_stability_flags")

,hypothesis,meaning_differs_across_cells,n
0,H1,NaN,102
1,H2,NaN,30
2,H3,NaN,84
3,H4,NaN,38
4,H5,NaN,22
5,H6,NaN,42


  saved table: results/stage11_refined_construct_analysis/v4_l12_granular_final_call49/notebook_analysis/10_contextual_validation/tables/pass_b_cell_stability_flags.csv  (318 rows)


## High vs low construct × high vs low rating (book-level)

In [4]:
frame = nh.load_refined_frame(cfg, "strict")
usable = frame[frame["analysable"].fillna(True)] if "analysable" in frame.columns else frame
constructs = [
    c
    for c in (
        "RAX_nonexplicit_affection",
        "RAX_explicit_sex",
        "RAX_h2_strict",
        "RAX_emotional_security",
        "RAX_status_display",
        "RAX_external_protection",
        "RAX_relational_darkness",
        "RARC",
    )
    if c in usable.columns
]

cell_rows = []
for c in constructs:
    q_lo, q_hi = usable[c].quantile(0.25), usable[c].quantile(0.75)
    for tier in ("high_rate", "low_rate"):
        for level, mask in (
            ("high_construct", usable[c] >= q_hi),
            ("low_construct", usable[c] <= q_lo),
        ):
            sub = usable.loc[mask & (usable["rating_class"] == tier), c]
            cell_rows.append(
                {
                    "construct": c,
                    "cell": f"{level}×{tier}",
                    "n_books": int(len(sub)),
                    "mean_share": float(sub.mean()) if len(sub) else float("nan"),
                }
            )
cells = pd.DataFrame(cell_rows)
display(cells)
ctx.save_table(cells, "construct_x_rating_cells")

print(
    "Interpretive questions for close reading (use human_review packets):\n"
    "- Is material provision in low-rated books mostly status expenditure?\n"
    "- Does high-rated explicit content co-occur with more aftercare/negotiation?\n"
    "- Does 'protection' in low-rated books look more like control?"
)

,construct,cell,n_books,mean_share
0,RAX_nonexplicit_affection,high_construct×high_rate,1302,0.2830
1,RAX_nonexplicit_affection,low_construct×high_rate,1199,0.1758
2,RAX_nonexplicit_affection,high_construct×low_rate,1352,0.2869
3,RAX_nonexplicit_affection,low_construct×low_rate,1444,0.1742
4,RAX_explicit_sex,high_construct×high_rate,1037,0.0093
5,RAX_explicit_sex,low_construct×high_rate,1221,0.0011
6,RAX_explicit_sex,high_construct×low_rate,1581,0.0132
7,RAX_explicit_sex,low_construct×low_rate,1432,0.0010
8,RAX_h2_strict,high_construct×high_rate,1292,0.0029
9,RAX_h2_strict,low_construct×high_rate,1188,0.0004


  saved table: results/stage11_refined_construct_analysis/v4_l12_granular_final_call49/notebook_analysis/10_contextual_validation/tables/construct_x_rating_cells.csv  (32 rows)
Interpretive questions for close reading (use human_review packets):
- Is material provision in low-rated books mostly status expenditure?
- Does high-rated explicit content co-occur with more aftercare/negotiation?
- Does 'protection' in low-rated books look more like control?
